# Campaign Timing and Seasonality Uplift

## 1) Problem framing
**Business question:** When should Lighthouse run campaigns to maximize donation volume and value?

- Explanatory goal: estimate month/quarter and campaign effects on donation amount.
- Predictive goal: predict whether a donation lands in the top-value quartile.
- Success metrics: explanatory `R2`/MAE and stable calendar effects; predictive `ROC-AUC`, `F1`, recall.

## 2-7) Lifecycle summary
- Data: `donations` (+ optional social/campaign context).
- Reproducible prep: temporal features and pipeline preprocessing.
- Modeling: linear regression (explanatory) + gradient boosting classifier (predictive).
- Evaluation: temporal holdout + CV + baseline comparison.
- Deployment candidate: `/api/reports/trends/campaign-seasonality` and dashboard campaign planner.


In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent]:
    if (candidate / 'data_loader.py').exists():
        sys.path.insert(0, str(candidate))
        break
from data_loader import load_table

don = load_table('donations').sort_values('donation_date').copy()
don['amount_effective'] = don['amount'].fillna(don['estimated_value']).clip(lower=0)
don['month'] = don['donation_date'].dt.month
don['quarter'] = don['donation_date'].dt.quarter.astype(str)
don['year'] = don['donation_date'].dt.year
don['is_year_end'] = don['month'].isin([10, 11, 12]).astype(int)
don['campaign_present'] = don['campaign_name'].notna().astype(int)
don['target_log_amount'] = np.log1p(don['amount_effective'])
don['high_value'] = (don['amount_effective'] >= don['amount_effective'].quantile(0.75)).astype(int)

features = ['month', 'quarter', 'year', 'is_year_end', 'campaign_present', 'campaign_name', 'channel_source', 'is_recurring', 'donation_type']
X = don[features]
y_reg = don['target_log_amount']
y_clf = don['high_value']

split = int(len(don) * 0.8)
Xtr, Xte = X.iloc[:split], X.iloc[split:]
ytr_reg, yte_reg = y_reg.iloc[:split], y_reg.iloc[split:]
ytr_clf, yte_clf = y_clf.iloc[:split], y_clf.iloc[split:]

cat_cols = [c for c in features if X[c].dtype == 'object' or X[c].dtype == 'bool']
num_cols = [c for c in features if c not in cat_cols]
prep = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

lin = Pipeline([('prep', prep), ('model', LinearRegression())])
lin.fit(Xtr, ytr_reg)
pred_reg = lin.predict(Xte)
print('Explanatory R2:', round(r2_score(yte_reg, pred_reg), 3))
print('Explanatory MAE:', round(mean_absolute_error(yte_reg, pred_reg), 3))

baseline = Pipeline([('prep', prep), ('model', DummyClassifier(strategy='prior'))])
gb = Pipeline([('prep', prep), ('model', GradientBoostingClassifier(random_state=42))])
baseline.fit(Xtr, ytr_clf)
gb.fit(Xtr, ytr_clf)
base_proba = baseline.predict_proba(Xte)[:, 1]
gb_proba = gb.predict_proba(Xte)[:, 1]
gb_pred = (gb_proba >= 0.5).astype(int)
print('Baseline AUC:', round(roc_auc_score(yte_clf, base_proba), 3))
print('GB AUC:', round(roc_auc_score(yte_clf, gb_proba), 3))
print('GB F1:', round(f1_score(yte_clf, gb_pred), 3))

cv = cross_validate(gb, X, y_clf, cv=5, scoring=['roc_auc', 'f1'])
print('CV AUC mean/std:', round(float(cv['test_roc_auc'].mean()), 3), round(float(cv['test_roc_auc'].std()), 3))

perm = permutation_importance(gb, Xte, yte_clf, n_repeats=8, random_state=42, scoring='roc_auc')
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(10)
print('\nTop predictive drivers:')
print(imp.round(4).to_string())

coef_values = np.ravel(lin.named_steps['model'].coef_)
coef_names = lin.named_steps['prep'].get_feature_names_out()
usable = min(len(coef_values), len(coef_names))
coef = pd.Series(coef_values[:usable], index=coef_names[:usable]).sort_values(key=lambda s: s.abs(), ascending=False).head(10)
print('\nTop explanatory effects:')
print(coef.round(4).to_string())

print('\nDecision recommendations: prioritize Q4 and top-performing channel/campaign combinations for fundraising windows.')


C:\Users\ethan\AppData\Local\Temp\ipykernel_10300\4056010529.py:23: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  don['amount_effective'] = don['amount'].fillna(don['estimated_value']).clip(lower=0)
C:\Users\ethan\AppData\Local\Temp\ipykerne

Explanatory R2: 0.843
Explanatory MAE: 0.518
Baseline AUC: 0.5
GB AUC: 0.789
GB F1: 0.577


CV AUC mean/std: 0.721 0.063



Top predictive drivers:
donation_type       0.1897
month               0.0776
is_year_end         0.0083
quarter             0.0022
campaign_present    0.0007
year                0.0000
campaign_name      -0.0026
channel_source     -0.0033
is_recurring       -0.0084

Top explanatory effects:
cat__donation_type_Monetary            2.6589
cat__donation_type_InKind              2.1225
cat__donation_type_SocialMedia        -2.0618
cat__donation_type_Skills             -1.5281
cat__donation_type_Time               -1.1915
num__month                             0.2527
cat__campaign_name_Year-End Hope      -0.2182
cat__quarter_1                         0.1823
cat__channel_source_PartnerReferral   -0.1293
cat__campaign_name_Summer of Safety    0.1237

Decision recommendations: prioritize Q4 and top-performing channel/campaign combinations for fundraising windows.


In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent]:
    if (candidate / 'trend_eval_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break
from trend_eval_helpers import print_calibration_bins, print_threshold_scan, bootstrap_linear_coefs, fairness_binary, fairness_regression_mae

print("\n=== Evaluation artifacts ===")
if yte_clf.nunique() >= 2:
    proba = gb.predict_proba(Xte)[:, 1]
    print_calibration_bins(yte_clf.values, proba)
    print_threshold_scan(yte_clf.values, proba)
    fairness_binary(gb, Xte, yte_clf, don.loc[Xte.index], ['quarter'], min_n=20)
bootstrap_linear_coefs(lin, Xtr, ytr_reg, n_boot=400, top_k=8)
